In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"

In [44]:
from concept_abstraction.training import train_ppo_model, SimpleQEstimator
from concept_abstraction.selection import greedy_selection_supervised, lp_selection_supervised, lp_selection_supervised_imperfect, multiple_selection_supervised
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import torch.nn as nn
from torchvision import models
import torch
from torchvision import transforms




In [4]:
is_jupyter = 'ipykernel' in sys.modules

In [5]:
if is_jupyter: 
    seed        = 42
    num_concepts_selected = 112
    out_folder = "cub"
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    num_concepts_selected = args.num_concepts_selected
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [6]:
results = {}
results['parameters'] = {'seed'      : seed,
        'num_concepts_selected': num_concepts_selected,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'num_concepts_selected': 112}


In [7]:
np.random.seed(seed)
random.seed(seed)

In [8]:
dataset = json.load(open("../../data/cub/preprocessed.json"))

In [9]:
def get_performance(selected_concepts,accuracy_by_concept):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
        max_iter=1000,  # increase if needed
        random_state=0,
        alpha=1e-3,  # instead of 0.0001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20
    )

    # Train the model
    mlp.fit(train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

## Perfrect Concepts

In [10]:
results['perfect'] = {}

In [11]:
train_X = np.array([row['attributes'] for row in dataset['train']])
test_X = np.array([row['attributes'] for row in dataset['test']])
train_Y = np.array([row['label'] for row in dataset['train']])
test_Y = np.array([row['label'] for row in dataset['test']])


In [12]:
# Denoising
total_X = np.concatenate([train_X,test_X])
total_Y = np.concatenate([train_Y,test_Y])

rows_by_label = {}
for label in set(total_Y):
    relevant_subset = total_X[total_Y == label]
    relevant_subset = np.round(np.mean(relevant_subset,axis=0))
    rows_by_label[label] = relevant_subset

    train_X[train_Y == label] = relevant_subset
    test_X[test_Y == label] = relevant_subset

In [55]:
results['perfect']['lp'] = {}
for num_concepts_selected in [10,20,30,40,50,60,70,80,90,100,110]:
    lp_concept_list = lp_selection_supervised(train_X,train_Y,num_concepts_selected)
    results['perfect']['lp'][num_concepts_selected] = {
        'reward': get_performance(lp_concept_list,np.ones(312)),
        'concepts': lp_concept_list
    }
    print("LP Performance {}: {}".format(num_concepts_selected,results['perfect']['lp'][num_concepts_selected] ))

LP Performance 10: {'reward': 0.6039005868139454, 'concepts': [6, 10, 20, 51, 54, 163, 218, 253, 289, 308]}
LP Performance 20: {'reward': 0.9437348981705213, 'concepts': [4, 6, 10, 20, 21, 51, 54, 75, 101, 149, 187, 193, 218, 236, 244, 249, 274, 283, 289, 311]}
LP Performance 30: {'reward': 0.9794615119088712, 'concepts': [6, 10, 20, 29, 36, 45, 51, 64, 75, 118, 126, 132, 149, 178, 187, 193, 218, 220, 227, 236, 240, 244, 249, 262, 274, 289, 293, 298, 309, 311]}
LP Performance 40: {'reward': 0.9744563341387642, 'concepts': [6, 29, 56, 64, 75, 90, 117, 125, 126, 132, 133, 134, 144, 145, 147, 149, 157, 166, 167, 183, 194, 208, 209, 210, 218, 220, 227, 228, 243, 244, 249, 268, 274, 277, 283, 284, 292, 299, 308, 309]}
LP Performance 50: {'reward': 0.9796341042457715, 'concepts': [6, 29, 56, 64, 75, 90, 91, 117, 120, 125, 126, 132, 133, 134, 144, 145, 147, 149, 157, 166, 167, 183, 194, 208, 209, 210, 218, 220, 227, 228, 243, 244, 249, 256, 268, 274, 277, 284, 292, 299, 308, 309]}
LP Performa

In [23]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]
results['perfect']['manual'] = {
    'reward': get_performance(manually_selected_concepts,np.ones(312)),
    'concepts': manually_selected_concepts
}
print("Manual Performance {}".format(results['perfect']['manual']['reward']))

Manual Performance 0.9794615119088712


#### Imperfect Concepts

In [24]:
img_locations = ["../../data/cub/images/{}".format(i['location']) for i in dataset['train']]
img_locations_test = ["../../data/cub/images/{}".format(i['location']) for i in dataset['test']]

In [25]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class CUBArrayDataset(Dataset):
    def __init__(self, img_locations, attributes, transform=None):
        """
        img_locations : list of image file paths
        attributes    : numpy array or torch tensor of shape (N, 312)
        transform     : torchvision transforms
        """
        self.img_locations = img_locations
        self.attributes = torch.tensor(attributes, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.img_locations)

    def __getitem__(self, idx):
        img = Image.open(self.img_locations[idx]).convert("RGB")
        label = self.attributes[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


In [26]:
resol = 224          # final crop size (change if needed)
resized_resol = 256  # initial resize before crop (if used)

transform = transforms.Compose([
    transforms.ColorJitter(brightness=32/255, saturation=(0.5, 1.5)),
    transforms.RandomResizedCrop(resol),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), # implicitly divides by 255
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[2, 2, 2])
])


In [27]:
from torch.utils.data import DataLoader

train_dataset = CUBArrayDataset(img_locations, train_X, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)


In [28]:

model = models.resnet50(pretrained=True)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 312),
    nn.Sigmoid()  # probabilities for each attribute
)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [29]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.BCELoss()  # Binary Cross Entropy for multi-label
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [30]:
threshold = 0.5  # probability cutoff for positive prediction
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_attrs = 0
    total_attrs   = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

        # ---- Average attribute accuracy ----
        preds = (outputs > threshold).float()      # shape: (batch, 312)
        correct_attrs += (preds == labels).sum().item()  # count correct per attribute
        total_attrs   += labels.numel()                  # total predictions = batch * 312

    epoch_loss = running_loss / len(train_loader.dataset)
    attr_accuracy = correct_attrs / total_attrs          # average across 312 attributes

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Loss: {epoch_loss:.4f}  "
          f"Attr-Accuracy (avg over 312): {attr_accuracy:.4f}")


Epoch [1/50]  Loss: 0.1358  Attr-Accuracy (avg over 312): 0.9492
Epoch [2/50]  Loss: 0.1216  Attr-Accuracy (avg over 312): 0.9535
Epoch [3/50]  Loss: 0.1181  Attr-Accuracy (avg over 312): 0.9547
Epoch [4/50]  Loss: 0.1135  Attr-Accuracy (avg over 312): 0.9562
Epoch [5/50]  Loss: 0.1112  Attr-Accuracy (avg over 312): 0.9568
Epoch [6/50]  Loss: 0.1088  Attr-Accuracy (avg over 312): 0.9576
Epoch [7/50]  Loss: 0.1068  Attr-Accuracy (avg over 312): 0.9583
Epoch [8/50]  Loss: 0.1044  Attr-Accuracy (avg over 312): 0.9589
Epoch [9/50]  Loss: 0.1034  Attr-Accuracy (avg over 312): 0.9595
Epoch [10/50]  Loss: 0.1008  Attr-Accuracy (avg over 312): 0.9601
Epoch [11/50]  Loss: 0.0993  Attr-Accuracy (avg over 312): 0.9607
Epoch [12/50]  Loss: 0.0978  Attr-Accuracy (avg over 312): 0.9612
Epoch [13/50]  Loss: 0.0961  Attr-Accuracy (avg over 312): 0.9617
Epoch [14/50]  Loss: 0.0947  Attr-Accuracy (avg over 312): 0.9623
Epoch [15/50]  Loss: 0.0926  Attr-Accuracy (avg over 312): 0.9629
Epoch [16/50]  Loss

In [31]:
from sklearn.metrics import f1_score

model.eval()
all_labels = []
all_preds  = []

with torch.no_grad():
    for imgs, labels in train_loader:  # or a separate val/test loader
        imgs  = imgs.to(device)
        labels = labels.cpu().numpy()           # (batch, 312)

        outputs = model(imgs).cpu().numpy()     # probabilities
        preds = (outputs > 0.5).astype(np.int32)  # threshold

        all_labels.append(labels)
        all_preds.append(preds)

# Stack all batches
all_labels = np.vstack(all_labels)  # shape (N, 312)
all_preds  = np.vstack(all_preds)   # shape (N, 312)

# F1 per attribute (macro across 312)
f1_macro = f1_score(all_labels, all_preds, average="macro")
print(f"Macro F1 (avg over 312 attributes): {f1_macro:.4f}")


Macro F1 (avg over 312 attributes): 0.4236


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [32]:
from torch.utils.data import DataLoader

train_pred_dataset = CUBArrayDataset(img_locations, train_X, transform=transform)
test_pred_dataset  = CUBArrayDataset(img_locations_test, 
                                     torch.zeros((len(img_locations_test), 312)),  # dummy labels
                                     transform=transform)

train_pred_loader = DataLoader(train_pred_dataset, batch_size=32, shuffle=False, num_workers=4)
test_pred_loader  = DataLoader(test_pred_dataset,  batch_size=32, shuffle=False, num_workers=4)


/tmp/ipykernel_662833/150385818.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.attributes = torch.tensor(attributes, dtype=torch.float32)


In [33]:

resol = 224  # same resolution as used in training
test_transform = transforms.Compose([
    transforms.CenterCrop(resol),
    transforms.ToTensor(),  # divides by 255
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[2, 2, 2])
])


In [34]:
train_pred_dataset = CUBArrayDataset(
    img_locations,
    torch.zeros((len(img_locations), 312)), 
    transform=test_transform
)
test_pred_dataset = CUBArrayDataset(
    img_locations_test,
    torch.zeros((len(img_locations_test), 312)),
    transform=test_transform
)

train_pred_loader = DataLoader(train_pred_dataset, batch_size=32, shuffle=False, num_workers=4)
test_pred_loader  = DataLoader(test_pred_dataset,  batch_size=32, shuffle=False, num_workers=4)


/tmp/ipykernel_662833/150385818.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.attributes = torch.tensor(attributes, dtype=torch.float32)


In [35]:
def get_predictions(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)           # probabilities after sigmoid
            preds.append(outputs.cpu().numpy())
    return np.vstack(preds)                 # shape: (N, 312)


In [36]:
pred_train_X = get_predictions(model, train_pred_loader, device)
pred_test_X  = get_predictions(model, test_pred_loader,  device)

print("pred_train_X:", pred_train_X.shape)
print("pred_test_X :", pred_test_X.shape)


pred_train_X: (5994, 312)
pred_test_X : (5794, 312)


In [37]:
from sklearn.neural_network import MLPClassifier

def get_performance_real(selected_concepts):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
        max_iter=1000,  # increase if needed
        random_state=0,
        alpha=1e-3,  # instead of 0.0001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20
    )

    # Train the model
    mlp.fit(pred_train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(pred_test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

In [38]:
results['imperfect'] = {}

In [39]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]


In [41]:
results['imperfect']['manual'] = {'reward': get_performance_real(manually_selected_concepts), 'concepts': manually_selected_concepts}

In [56]:
results['imperfect']['lp'] = {}
for c in results['perfect']['lp']:
    results['imperfect']['lp'][c] = {
        'reward': get_performance_real(results['perfect']['lp'][c]['concepts']),
        'concepts': c
    }

In [58]:
results['imperfect']['multiple_lp'] = {}
for c in results['perfect']['lp']:
    imperfect_concepts = multiple_selection_supervised(train_X,train_Y,c)
    results['imperfect']['multiple_lp'][c] = {
        'reward': get_performance_real(imperfect_concepts), 
        'concepts': imperfect_concepts
    }
results['imperfect']['multiple_lp']

{10: {'reward': 0.3850535036244391,
  'concepts': [6, 20, 51, 54, 151, 209, 218, 235, 244, 289]},
 20: {'reward': 0.4815326199516741,
  'concepts': [6,
   7,
   20,
   35,
   51,
   54,
   117,
   132,
   149,
   151,
   163,
   209,
   218,
   235,
   236,
   244,
   259,
   289,
   304,
   308]},
 30: {'reward': 0.5329651363479462,
  'concepts': [6,
   7,
   10,
   14,
   20,
   35,
   51,
   54,
   69,
   117,
   131,
   132,
   149,
   151,
   163,
   178,
   193,
   209,
   218,
   220,
   235,
   236,
   240,
   244,
   253,
   259,
   260,
   289,
   304,
   308]},
 40: {'reward': 0.5348636520538488,
  'concepts': [6,
   7,
   10,
   14,
   20,
   21,
   25,
   35,
   45,
   51,
   54,
   69,
   101,
   116,
   117,
   131,
   132,
   149,
   151,
   163,
   178,
   193,
   194,
   209,
   218,
   220,
   235,
   236,
   240,
   244,
   249,
   253,
   254,
   259,
   260,
   274,
   289,
   304,
   308,
   311]},
 50: {'reward': 0.5447014152571625,
  'concepts': [6,
   7,
   10

## Save Data

In [66]:
save_path = get_save_path(out_folder,save_name)

In [67]:
delete_duplicate_results(out_folder,"",results)

In [68]:
json.dump(results,open('../../results/'+save_path,'w'))